# LeetCode 3307 — Find the K-th Character in String Game II bằng Python

- Link: https://leetcode.com/problems/find-the-k-th-character-in-string-game-ii/
- Độ khó: **Hard**
- Chủ đề: **Đệ quy theo nửa** (độ dài gấp đôi mỗi bước), **biểu diễn nhị phân của `k − 1`**: bit `i` bằng 1 ⇔ ký tự nằm ở nửa sau của bước `i`, cộng dịch nếu `operations[i] = 1`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kthCharacter(self, k: int, operations: List[int]) -> str:
```

## 1. Mục tiêu học

1. Nhận ra sau `m` thao tác, độ dài là `2^m` — không thể dựng chuỗi khi `m = 100`, nhưng `k ≤ 10^14 < 2^47` nên chỉ **47 thao tác đầu** có ý nghĩa.
2. Vị trí `p = k − 1` (đếm từ 0): ở bước `i`, nửa sau là bản sao (có thể dịch) của nửa trước ⇒ nếu bit `i` của `p` bằng 1, ký tự là bản sao của ký tự ở `p − 2^i`, cộng 1 nếu `operations[i] = 1`.
3. Đáp án: `chr(ord('a') + (Σ_{i: bit i của p = 1} operations[i]) mod 26)`.
4. Ba cách: mô phỏng (nhỏ), đệ quy theo nửa, đếm bit.
5. Kiểm chứng bằng mô phỏng.

## 2. Đề bài & ví dụ

Ban đầu `word = "a"`. Thực hiện lần lượt các thao tác `operations[i]`:
- `0`: nối thêm một bản sao của `word` vào cuối;
- `1`: tạo chuỗi mới bằng cách đổi mỗi ký tự thành ký tự **kế tiếp** (`'z' → 'a'`) rồi nối vào cuối `word`.

Trả về ký tự thứ `k` (đếm từ 1) của `word` sau mọi thao tác.

**Ví dụ 1:** `k = 5`, `operations = [0,0,0]` → `"a"`. **Ví dụ 2:** `k = 10`, `operations = [0,1,0,1]` → `"b"` (`word = "aabbaabbbbccbbcc"`).

**Ràng buộc:** `1 ≤ k ≤ 10^14`; `1 ≤ len(operations) ≤ 100`; đề bảo đảm `word` có ít nhất `k` ký tự.

In [1]:
from typing import List

examples = [(5, [0, 0, 0], "a"), (10, [0, 1, 0, 1], "b")]
for k, ops, exp in examples:
    print('k =', k, 'ops =', ops, '-> kỳ vọng', repr(exp))

k = 5 ops = [0, 0, 0] -> kỳ vọng 'a'
k = 10 ops = [0, 1, 0, 1] -> kỳ vọng 'b'


## 3. Kiến thức nền

### 3.1. Mô phỏng (chỉ cho chuỗi ngắn)

In [2]:
def simulate(ops, limit=1 << 16):
    w = "a"
    for op in ops:
        if len(w) >= limit:
            break
        w = w + (w if op == 0 else ''.join(chr((ord(c) - 97 + 1) % 26 + 97) for c in w))
    return w

print(simulate([0, 1, 0, 1]))

aabbaabbbbccbbcc


### 3.2. Cấu trúc nửa trước / nửa sau

Sau bước `i` (tính từ 0), `word` dài `2^{i+1}`: nửa đầu `[0, 2^i)` là `word` trước bước đó, nửa sau `[2^i, 2^{i+1})` là bản sao (dịch 1 nếu `ops[i] = 1`). Vị trí `p` trong nửa sau ⇔ bit `i` của `p` bằng 1 (khi `p < 2^{i+1}`).

### 3.3. Chỉ cần đủ bit của `k − 1`

Ký tự thứ `k` đã cố định ngay khi độ dài `≥ k`; các thao tác sau chỉ nối thêm phía sau. Số bit của `k − 1` là `≤ 47` với `k ≤ 10^14`.

In [3]:
print('bit_length của 10^14 - 1:', (10 ** 14 - 1).bit_length())
print('p = 9 = ', bin(9), '-> bit 0 và bit 3 bằng 1')

bit_length của 10^14 - 1: 47
p = 9 =  0b1001 -> bit 0 và bit 3 bằng 1


### 3.4. Nhìn tận mắt: vị trí, biểu diễn nhị phân và độ dịch

Bảng dưới in từng vị trí `p` của chuỗi sau 4 thao tác `[0, 1, 0, 1]`, biểu diễn nhị phân 4 bit của `p` (bit 0 ở **bên phải**), các bit "được tính" (bit 1 tại thao tác loại 1) và ký tự. Quan sát: ký tự chỉ phụ thuộc **tập bit 1 rơi vào thao tác loại 1** — thứ tự hay số lượng bit 1 rơi vào thao tác loại 0 không ảnh hưởng gì.

In [4]:
ops_demo = [0, 1, 0, 1]
w_demo = simulate(ops_demo)
for p in range(len(w_demo)):
    bits = format(p, '04b')
    counted = [i for i in range(4) if p >> i & 1 and ops_demo[i] == 1]
    print(f"p = {p:2d}  {bits}  bit tính dịch: {str(counted):8s} -> dịch {len(counted)} -> {w_demo[p]!r}")

p =  0  0000  bit tính dịch: []       -> dịch 0 -> 'a'
p =  1  0001  bit tính dịch: []       -> dịch 0 -> 'a'
p =  2  0010  bit tính dịch: [1]      -> dịch 1 -> 'b'
p =  3  0011  bit tính dịch: [1]      -> dịch 1 -> 'b'
p =  4  0100  bit tính dịch: []       -> dịch 0 -> 'a'
p =  5  0101  bit tính dịch: []       -> dịch 0 -> 'a'
p =  6  0110  bit tính dịch: [1]      -> dịch 1 -> 'b'
p =  7  0111  bit tính dịch: [1]      -> dịch 1 -> 'b'
p =  8  1000  bit tính dịch: [3]      -> dịch 1 -> 'b'
p =  9  1001  bit tính dịch: [3]      -> dịch 1 -> 'b'
p = 10  1010  bit tính dịch: [1, 3]   -> dịch 2 -> 'c'
p = 11  1011  bit tính dịch: [1, 3]   -> dịch 2 -> 'c'
p = 12  1100  bit tính dịch: [3]      -> dịch 1 -> 'b'
p = 13  1101  bit tính dịch: [3]      -> dịch 1 -> 'b'
p = 14  1110  bit tính dịch: [1, 3]   -> dịch 2 -> 'c'
p = 15  1111  bit tính dịch: [1, 3]   -> dịch 2 -> 'c'


Hệ quả thú vị: các vị trí có **cùng tập bit ở các thao tác loại 1** luôn mang cùng ký tự — ví dụ `p = 0, 1, 4, 5` đều là `'a'` vì chúng chỉ khác nhau ở bit 0 và bit 2 (đều là thao tác loại 0).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `k = 10` ⇒ `p = 9 = 1001₂`, `ops = [0, 1, 0, 1]`.

- Bit 3 = 1 ⇒ ở bước 3 (dài 16), vị trí 9 thuộc nửa sau; `ops[3] = 1` ⇒ dịch 1. Quay về vị trí `9 − 8 = 1`.
- Bit 2 = 0, bit 1 = 0 ⇒ ở nửa đầu, không dịch.
- Bit 0 = 1 ⇒ vị trí 1 thuộc nửa sau của bước 0; `ops[0] = 0` ⇒ không dịch. Về vị trí 0 = `'a'`.

Tổng dịch 1 ⇒ `'b'` ✓.

## 5. Cách 1 — Brute Force: dựng chuỗi

In [5]:
def brute(k, ops):
    w = "a"
    for op in ops:
        if len(w) >= k:
            break
        w = w + (w if op == 0 else ''.join(chr((ord(c) - 97 + 1) % 26 + 97) for c in w))
    return w[k - 1]

for k, ops, exp in examples:
    print(repr(brute(k, ops)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'b' | kỳ vọng 'b'


Dừng sớm khi đủ `k` ký tự giúp tránh chuỗi `2^100`, nhưng vẫn tốn `O(k)` bộ nhớ — không chạy được với `k = 10^14`.

## 6. Ý tưởng thuật toán / bước đột phá

```
p = k − 1
shift = Σ_{i < len(ops), bit i của p = 1} ops[i]
return chr(97 + shift % 26)
```

`O(log k)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy theo nửa (từ bước lớn nhất xuống)

In [6]:
def recursive(k, ops):
    def go(pos, step):                     # ký tự ở vị trí pos (0-index) sau các bước 0..step-1, trả về số lần dịch
        if step == 0:
            return 0
        half = 1 << (step - 1)
        if pos < half:
            return go(pos, step - 1)
        return go(pos - half, step - 1) + ops[step - 1]
    p = k - 1
    m = max(p.bit_length(), 0)             # chỉ cần m bước đầu
    return chr(97 + go(p, m) % 26)

for k, ops, exp in examples:
    print(repr(recursive(k, ops)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'b' | kỳ vọng 'b'


Độ sâu đệ quy `≤ 47` — an toàn.

### 7.2. Cách 3 — Đếm bit (lời giải chính)

In [7]:
def bit_count(k, ops):
    p = k - 1
    shift = 0
    i = 0
    while p:
        if p & 1:
            shift += ops[i]
        p >>= 1
        i += 1
    return chr(97 + shift % 26)

for k, ops, exp in examples:
    print(repr(bit_count(k, ops)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'b' | kỳ vọng 'b'


### 7.3. Cách 4 — Lặp từ trên xuống bằng độ dài (không dùng phép toán bit)

Giữ `length = 2^m` với `m` nhỏ nhất để `length ≥ k`; ở mỗi bước nếu `k > length/2` thì `k −= length/2` và cộng `ops[bước]`. Cách viết "tự nhiên" mà nhiều người nghĩ ra trước.

In [8]:
def top_down(k, ops):
    m, length = 0, 1
    while length < k:
        length *= 2; m += 1
    shift = 0
    for step in range(m - 1, -1, -1):
        half = length // 2
        if k > half:
            k -= half
            shift += ops[step]
        length = half
    return chr(97 + shift % 26)

for k, ops, exp in examples:
    print(repr(top_down(k, ops)), '| kỳ vọng', repr(exp))

'a' | kỳ vọng 'a'
'b' | kỳ vọng 'b'


## 8. TRACE từng bước

Trace cách 3 với `k = 23`, `ops = [1, 0, 1, 1, 0, 1]`.

In [9]:
def trace(k, ops):
    p = k - 1
    print(f'p = k - 1 = {p} = {bin(p)}')
    shift = 0
    for i in range(p.bit_length()):
        b = p >> i & 1
        add = ops[i] if b else 0
        shift += add
        print(f'  bit {i} = {b}, ops[{i}] = {ops[i]} -> cộng {add}, tổng dịch = {shift}')
    print('đáp án:', chr(97 + shift % 26), '| mô phỏng:', simulate(ops)[k - 1])

trace(23, [1, 0, 1, 1, 0, 1])

p = k - 1 = 22 = 0b10110
  bit 0 = 0, ops[0] = 1 -> cộng 0, tổng dịch = 0
  bit 1 = 1, ops[1] = 0 -> cộng 0, tổng dịch = 0
  bit 2 = 1, ops[2] = 1 -> cộng 1, tổng dịch = 1
  bit 3 = 0, ops[3] = 1 -> cộng 0, tổng dịch = 1
  bit 4 = 1, ops[4] = 0 -> cộng 0, tổng dịch = 1
đáp án: b | mô phỏng: b


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def kthCharacter(self, k: int, operations: List[int]) -> str:
        p = k - 1
        shift = 0
        i = 0
        while p:
            if p & 1 and operations[i]:
                shift += 1
            p >>= 1
            i += 1
        return chr(ord('a') + shift % 26)

for k, ops, exp in examples:
    got = Solution().kthCharacter(k, ops)
    print(repr(got), 'OK' if got == exp else 'SAI')

'a' OK
'b' OK


Truy cập `operations[i]` an toàn vì đề bảo đảm độ dài cuối `≥ k`, tức `len(operations) ≥ (k − 1).bit_length()`.

## 10. Vì sao thuật toán đúng

**Bổ đề:** với `p < 2^m`, ký tự ở vị trí `p` sau `m` bước bằng `'a'` dịch `Σ_{i < m, bit i của p} ops[i]`.

Quy nạp theo `m`. `m = 0`: `p = 0`, ký tự `'a'`, tổng rỗng. Bước `m − 1 → m`: nếu bit `m − 1` của `p` bằng 0 thì `p < 2^{m−1}` thuộc nửa đầu (không đổi); nếu bằng 1 thì `p` thuộc nửa sau, là ký tự ở `p − 2^{m−1}` (cùng các bit thấp) dịch thêm `ops[m − 1]`. Áp dụng giả thiết quy nạp cho `p` hoặc `p − 2^{m−1}`.

Ký tự thứ `k` không đổi sau khi độ dài `≥ k` ⇒ chỉ cần `m = bit_length(k − 1)` bước.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dựng chuỗi | `O(k)` | `O(k)` |
| 2. Đệ quy theo nửa | `O(log k)` | `O(log k)` |
| 3. Đếm bit | `O(log k)` | `O(1)` |
| 4. Lặp theo độ dài | `O(log k)` | `O(1)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'recursive': recursive, 'bit_count': bit_count, 'top_down': top_down,
         'submit': lambda k, o: Solution().kthCharacter(k, o)}
edge = [(1, [1], 'a'), (2, [1], 'b'), (2, [0], 'a'), (4, [1, 1], 'c'), (8, [1, 1, 1], 'd'), (3, [0, 1], 'b')]
for k, ops, exp in examples + edge:
    assert brute(k, ops) == exp, (k, ops, brute(k, ops))
    for name, f in impls.items():
        assert f(k, ops) == exp, (name, k, ops)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [12]:
import random
rng = random.Random(3307)
cnt = 0
for _ in range(1000):
    m = rng.randint(1, 14)
    ops = [rng.randint(0, 1) for _ in range(m)]
    k = rng.randint(1, 1 << m)
    exp = brute(k, ops)
    for name, f in impls.items():
        assert f(k, ops) == exp, (name, k, ops)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test.')

Khớp mô phỏng trên 1000 bộ test.


### 12.3. Ca lớn `k = 10^14`, 100 thao tác

Kiểm tra độc lập: với `ops` toàn 1, dịch = số bit 1 của `k − 1` (popcount); với `ops` toàn 0, luôn `'a'`.

In [13]:
import time
K = 10 ** 14
t = time.time()
for _ in range(10000):
    Solution().kthCharacter(K, [rng.randint(0, 1) for _ in range(100)])
print(f'10^4 truy vấn k = 10^14: {time.time() - t:.2f}s')
for k in [K, K - 1, 2 ** 46, 2 ** 46 + 1, 123456789012345]:
    assert Solution().kthCharacter(k, [1] * 100) == chr(97 + bin(k - 1).count('1') % 26)
    assert Solution().kthCharacter(k, [0] * 100) == 'a'
print('ops toàn 1 -> popcount(k-1) mod 26; ops toàn 0 -> a: đúng.')

10^4 truy vấn k = 10^14: 0.40s
ops toàn 1 -> popcount(k-1) mod 26; ops toàn 0 -> a: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng mọi thao tác loại 1 (không xét bit)

In [14]:
def wrong_all_ops(k, ops):
    m = (k - 1).bit_length()
    return chr(97 + sum(ops[:m]) % 26)                        # SAI

print('sai :', wrong_all_ops(10, [0, 1, 0, 1]), '| đúng:', Solution().kthCharacter(10, [0, 1, 0, 1]))

sai : c | đúng: b


**Vì sao sai:** chỉ những bước mà vị trí nằm ở **nửa sau** mới dịch. Ở ví dụ 2, bước 1 (`ops[1] = 1`) không ảnh hưởng vì bit 1 của `9` bằng 0.

### Lỗi 2 — Dùng bit của `k` thay vì `k − 1`

In [15]:
def wrong_k_not_k_minus_1(k, ops):
    p, shift, i = k, 0, 0                                    # SAI: phải là k - 1
    while p and i < len(ops):
        if p & 1:
            shift += ops[i]
        p >>= 1; i += 1
    return chr(97 + shift % 26)

print('sai :', wrong_k_not_k_minus_1(10, [0, 1, 0, 1]), '| đúng:', Solution().kthCharacter(10, [0, 1, 0, 1]))

sai : c | đúng: b


**Vì sao sai:** `k` đếm từ 1; phân tích nửa trước/sau dựa trên chỉ số từ 0. `10 = 1010₂` cho bit 1, 3; `9 = 1001₂` cho bit 0, 3.

### Lỗi 3 — Quên `mod 26` khi dịch

In [16]:
def wrong_no_mod(k, ops):
    p, shift, i = k - 1, 0, 0
    while p:
        if p & 1:
            shift += ops[i]
        p >>= 1; i += 1
    return chr(97 + shift)                                    # SAI: có thể vượt 'z'

k_bad = 2 ** 30                     # k - 1 có 30 bit 1
print('sai :', repr(wrong_no_mod(k_bad, [1] * 40)), '| đúng:', repr(Solution().kthCharacter(k_bad, [1] * 40)))

sai : '\x7f' | đúng: 'e'


**Vì sao sai:** dịch 30 lần vượt quá bảng chữ cái; `'z'` sau đó phải quay về `'a'`.

### Lỗi 4 — Ghép bit với thao tác theo thứ tự ngược

In [17]:
def wrong_reversed(k, ops):
    p = k - 1
    m = p.bit_length()
    shift = sum(ops[m - 1 - i] for i in range(m) if p >> i & 1)   # SAI: bit thấp ứng với thao tác cuối
    return chr(97 + shift % 26)

print('sai :', wrong_reversed(3, [0, 1]), '| đúng:', Solution().kthCharacter(3, [0, 1]))

sai : a | đúng: b


**Vì sao sai:** bit thấp nhất (bit 0) ứng với **thao tác đầu tiên** (khi chuỗi còn dài 1 → 2). `word = "aabb"`, ký tự thứ 3 là `'b'`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi/mảng **gấp đôi** mỗi bước ⇒ vị trí tương ứng với biểu diễn nhị phân; truy ngược bằng "nửa sau ⇒ trừ nửa".
- `k` cực lớn (`10^14`, `10^18`) với cấu trúc đệ quy ⇒ không bao giờ dựng dữ liệu.
- Cẩn thận chỉ số 0/1.
- Các bài "chuỗi tự sinh" cùng họ: LC 779 (K-th Symbol in Grammar — dãy Thue–Morse, đáp án là `popcount(k − 1) mod 2`), LC 1545 (Find Kth Bit in Nth Binary String — nửa sau là bản **đảo + lật bit**). Cả ba đều giải bằng cùng một khuôn: xác định vị trí nằm ở nửa nào, ánh xạ về nửa đầu, cộng dồn "hiệu ứng" của phép biến đổi.
- Khi đề cho số thao tác lớn (100) nhưng `k` nhỏ hơn hẳn `2^100`, hãy tự hỏi: **thao tác nào thật sự ảnh hưởng** tới vị trí cần hỏi? Ở đây chỉ các thao tác có chỉ số `< bit_length(k − 1)`.

## 15. Biến thể của bài

1. **LC 3304 (bản I):** mọi thao tác đều loại 1, `k ≤ 500` ⇒ `popcount(k − 1)`.
2. Thao tác loại 2: nối bản **đảo ngược** ⇒ nửa sau ánh xạ `p → 2^{i+1} − 1 − p`.
3. Dịch theo bước `d_i` khác nhau ⇒ `Σ d_i` trên các bit 1.
4. Trả về **số lần xuất hiện** của một ký tự trong `k` ký tự đầu.

In [18]:
# Biến thể 2 và 3
def reverse_variant(k, ops):
    """ops[i] = 0: nối bản sao, 1: nối bản dịch 1, 2: nối bản đảo ngược."""
    p = k - 1
    m = p.bit_length()
    shift = 0
    for i in range(m - 1, -1, -1):
        half = 1 << i
        if p >= half:
            if ops[i] == 2:
                p = 2 * half - 1 - p
            else:
                p -= half
                shift += ops[i]
    return chr(97 + shift % 26)

def reverse_brute(k, ops):
    w = "a"
    for op in ops:
        if len(w) >= k:
            break
        if op == 0:
            w += w
        elif op == 1:
            w += ''.join(chr((ord(c) - 96) % 26 + 97) for c in w)
        else:
            w += w[::-1]
    return w[k - 1]

def step_shift(k, d):
    p, s, i = k - 1, 0, 0
    while p:
        if p & 1:
            s += d[i]
        p >>= 1; i += 1
    return chr(97 + s % 26)

rng = random.Random(15)
for _ in range(500):
    m = rng.randint(1, 10)
    ops = [rng.randint(0, 2) for _ in range(m)]
    k = rng.randint(1, 1 << m)
    assert reverse_variant(k, ops) == reverse_brute(k, ops)
    d = [rng.randint(0, 30) for _ in range(m)]
    w = "a"
    for di in d:
        w += ''.join(chr((ord(c) - 97 + di) % 26 + 97) for c in w)
    assert step_shift(k, d) == w[k - 1]
print('Biến thể 2 (đảo ngược) và 3 (dịch d_i) khớp mô phỏng.')

Biến thể 2 (đảo ngược) và 3 (dịch d_i) khớp mô phỏng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Dựng chuỗi | `O(k)` | Oracle |
| 2 | Đệ quy theo nửa | `O(log k)` | Trực quan |
| 3 | Đếm bit | `O(log k)` | **Nộp bài** |
| 4 | Lặp theo độ dài | `O(log k)` | Không cần bit |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `k = 1`, `ops = [1, 1, 1]`.

**Bài 2 (tính tay):** `k = 6`, `ops = [1, 0, 1]`.

**Bài 3 (điền TODO):** trả về `k` ký tự **đầu tiên** (khi `k ≤ 10^5`) mà không dựng thừa.

**Bài 4 (tự viết lại):** biến thể 4 — đếm số ký tự `'a'` trong `k` ký tự đầu (`k ≤ 10^14`).

**Bài 5 (mini challenge):** tìm `k` **nhỏ nhất** sao cho ký tự thứ `k` là `c` cho trước (hoặc `-1`).

In [19]:
def prefix_todo(k, ops):
    # TODO
    raise NotImplementedError

def count_a_todo(k, ops):
    # TODO
    raise NotImplementedError

def first_position_todo(c, ops):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ký tự đầu không bao giờ đổi. (2) `p = 0`. (3) `'a'`.

**Bài 2.** (1) `p = 5 = 101₂`. (2) Bit 0 (`ops[0] = 1`) và bit 2 (`ops[2] = 1`). (3) `'c'`.

**Bài 3.** (1) `[kthCharacter(i) for i in 1..k]` là `O(k log k)`. (2) Hoặc dựng dần chuỗi tới độ dài `≥ k`. (3) Cắt `k` ký tự.

**Bài 4.** (1) Đếm `cnt[m][s]` = số vị trí trong `2^m` đầu có dịch `≡ s (mod 26)` — gấp đôi mỗi bước. (2) Tách `k` theo bit như "đếm số `< k`". (3) Vét cạn nhỏ.

**Bài 5.** (1) Vị trí nhỏ nhất có tổng dịch `≡ c − 'a' (mod 26)`. (2) Duyệt `p` tăng dần là chậm; dùng DP trên tập con bit (chọn tập bit có `ops = 1` với tổng mong muốn, các bit `ops = 0` để 0). (3) Chọn các bit nhỏ nhất có `ops = 1` để đạt số lượng cần (mod 26).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().kthCharacter(1, [1, 1, 1]) == 'a'
assert Solution().kthCharacter(6, [1, 0, 1]) == 'c' == brute(6, [1, 0, 1])

def prefix(k, ops):
    w = "a"
    for op in ops:
        if len(w) >= k:
            break
        w += w if op == 0 else ''.join(chr((ord(c) - 96) % 26 + 97) for c in w)
    return w[:k]

assert prefix(10, [0, 1, 0, 1]) == "aabbaabbbb"
print('Bài 1: a | Bài 2: c | Bài 3:', prefix(10, [0, 1, 0, 1]))

Bài 1: a | Bài 2: c | Bài 3: aabbaabbbb


In [21]:
# Bài 4, 5
def count_a(k, ops):
    # cnt[s] cho khối 2^i đầu: số vị trí có tổng dịch ≡ s
    blocks = [[1] + [0] * 25]
    for i in range(min(len(ops), (k - 1).bit_length() + 1)):
        prev = blocks[-1]
        sh = ops[i]
        blocks.append([prev[s] + prev[(s - sh) % 26] for s in range(26)])
    # đếm vị trí p < k: duyệt bit từ cao xuống
    total = [0] * 26
    acc = 0                                    # tổng dịch của tiền tố bit đã chọn = 1
    m = k.bit_length()
    for i in range(m - 1, -1, -1):
        if k >> i & 1:
            blk = blocks[i]                    # đặt bit i = 0, các bit thấp tự do
            for s in range(26):
                total[(s + acc) % 26] += blk[s]
            acc += ops[i] if i < len(ops) else 0
    return total[0]

def first_position(c, ops):
    need = (ord(c) - 97) % 26
    ones = [i for i, o in enumerate(ops) if o == 1]
    if need == 0:
        return 1
    if len(ones) < need:
        return -1
    p = sum(1 << i for i in ones[:need])
    return p + 1

rng = random.Random(4)
for _ in range(300):
    m = rng.randint(1, 10)
    ops = [rng.randint(0, 1) for _ in range(m)]
    k = rng.randint(1, 1 << m)
    w = prefix(1 << m, ops)
    assert count_a(k, ops) == w[:k].count('a'), (k, ops)
    c = chr(97 + rng.randint(0, 5))
    exp = w.find(c) + 1 if c in w else -1
    assert first_position(c, ops) == exp, (c, ops)
print('Bài 4, 5 khớp mô phỏng.')

Bài 4, 5 khớp mô phỏng.


Ghi chú bài 5: với `need < 26` (không cần quay vòng), vị trí nhỏ nhất dùng đúng `need` bit 1 thấp nhất trong các vị trí `ops = 1`; nếu không đủ thì ký tự `c` không xuất hiện (với `ops` dài ≤ 100 và `need ≤ 25`, quay vòng chỉ xảy ra khi có ≥ 26 bit — vẫn lớn hơn phương án `need` bit).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
p = k - 1; shift = 0; i = 0
while p:
    if p & 1: shift += ops[i]
    p >>= 1; i += 1
return chr(ord('a') + shift % 26)
```

**Checklist:**

- [ ] Dùng `k − 1` (chỉ số từ 0)?
- [ ] Bit `i` ↔ thao tác `i` (bit thấp = thao tác đầu)?
- [ ] Chỉ cộng `ops[i]` khi bit `i` bằng 1?
- [ ] `mod 26`?
- [ ] Không dựng chuỗi?